# Value Extensions

Runnable samples for every method in `CSharpHelperExtensions.Values`.  
Run the **Setup** cell first, then any section independently.

| Section | Methods |
|---|---|
| [1. In](#1-in) | `In<T>` |
| [2. IsBetween](#2-isbetween) | `IsBetween<T>` · `BetweenComparison` |
| [3. ToJson](#3-tojson) | `ToJson<T>` |
| [4. Chaining Examples](#4-chaining-examples) | Composing value extensions |

## Setup

> **Run this cell first.** It loads the compiled library and imports the required namespace.
>
> Build first if the DLL is missing: `dotnet build` from the repo root.

In [ ]:
#r "../src/CSharpHelperExtensions/bin/Debug/net10.0/CSharpHelperExtensions.dll"
using CSharpHelperExtensions.Values;  // In, IsBetween, BetweenComparison, ToJson

---
## 1. In

| Method | Signature | Returns |
|---|---|---|
| `In<T>` | `T, params T[] → bool` | `true` when `value` equals any item in the set (like SQL `IN`) |

Returns `false` when no candidates are supplied. Equality uses the default comparer for `T`.

In [ ]:
// String membership — role check
display("admin".In("admin", "superadmin"));   // True
display("guest".In("admin", "superadmin"));   // False
display("superadmin".In("admin", "superadmin")); // True

In [ ]:
// Integer membership
display(3.In(1, 2, 3, 4));   // True
display(5.In(1, 2, 3, 4));   // False
display(1.In(1, 2, 3, 4));   // True  (first element)

In [ ]:
// Empty set — always false
display("x".In());           // False  (no candidates supplied)
display(42.In());             // False

In [ ]:
// Enum membership
enum HttpMethod { Get, Post, Put, Delete, Patch }

var method = HttpMethod.Post;

display(method.In(HttpMethod.Post, HttpMethod.Put, HttpMethod.Patch));  // True  (mutating methods)
display(method.In(HttpMethod.Get, HttpMethod.Delete));                   // False
display(HttpMethod.Get.In(HttpMethod.Get, HttpMethod.Post));             // True

---
## 2. IsBetween

| `BetweenComparison` mode | Behaviour |
|---|---|
| `None` (default) | Inclusive on both ends: `lower ≤ value ≤ upper` |
| `ExcludeBoth` | Exclusive on both ends: `lower < value < upper` |
| `ExcludeLower` | Exclusive lower, inclusive upper: `lower < value ≤ upper` |
| `ExcludeUpper` | Inclusive lower, exclusive upper: `lower ≤ value < upper` |

Works on any type that implements `IComparable<T>` — integers, `DateTime`, `string`, etc.

In [ ]:
// BetweenComparison.None (default) — inclusive on both ends
display(5.IsBetween(1, 10));    // True   (interior)
display(1.IsBetween(1, 10));    // True   (lower bound included)
display(10.IsBetween(1, 10));   // True   (upper bound included)
display(0.IsBetween(1, 10));    // False  (below lower)
display(11.IsBetween(1, 10));   // False  (above upper)

In [ ]:
// BetweenComparison.ExcludeBoth — exclusive on both ends
display(5.IsBetween(1, 10, BetweenComparison.ExcludeBoth));    // True   (interior)
display(1.IsBetween(1, 10, BetweenComparison.ExcludeBoth));    // False  (lower bound excluded)
display(10.IsBetween(1, 10, BetweenComparison.ExcludeBoth));   // False  (upper bound excluded)

In [ ]:
// BetweenComparison.ExcludeLower — exclusive lower, inclusive upper
display(5.IsBetween(1, 10, BetweenComparison.ExcludeLower));    // True
display(1.IsBetween(1, 10, BetweenComparison.ExcludeLower));    // False  (lower excluded)
display(10.IsBetween(1, 10, BetweenComparison.ExcludeLower));   // True   (upper included)

In [ ]:
// BetweenComparison.ExcludeUpper — inclusive lower, exclusive upper
display(5.IsBetween(1, 10, BetweenComparison.ExcludeUpper));    // True
display(1.IsBetween(1, 10, BetweenComparison.ExcludeUpper));    // True   (lower included)
display(10.IsBetween(1, 10, BetweenComparison.ExcludeUpper));   // False  (upper excluded)

In [ ]:
// Works on any IComparable<T> — DateTime and string

// DateTime range check
var start = new DateTime(2024, 1, 1);
var end   = new DateTime(2024, 12, 31);
var mid   = new DateTime(2024, 6, 15);

display(mid.IsBetween(start, end));                               // True
display(start.IsBetween(start, end));                             // True  (inclusive)
display(start.IsBetween(start, end, BetweenComparison.ExcludeBoth)); // False  (excluded)

// String lexicographic range
display("mango".IsBetween("apple", "orange"));                    // True
display("zebra".IsBetween("apple", "orange"));                    // False  (lexically after "orange")

---
## 3. ToJson

| Parameter | Type | Default | Notes |
|---|---|---|---|
| `value` | `T` | — | Object to serialize; returns `null` when this is `null` |
| `indentation` | `bool` | `false` | `true` → pretty-printed; `false` → compact single-line |

Serializes via **Newtonsoft.Json**. Works on both reference types and value types.

In [ ]:
// Compact output (default)
display(new { Name = "Alice", Age = 30 }.ToJson());   // {"Name":"Alice","Age":30}

// With a list
display(new { Tags = new[] { "a", "b", "c" }, Active = true }.ToJson());
// {"Tags":["a","b","c"],"Active":true}

In [ ]:
// Indented (pretty-printed) output
display(new { Name = "Alice", Age = 30 }.ToJson(indentation: true));
// {
//   "Name": "Alice",
//   "Age": 30
// }

In [ ]:
// Value types — works directly on int, bool, DateTime, etc.
display(42.ToJson());                 // 42
display(true.ToJson());              // true
display(3.14.ToJson());              // 3.14
display(DateTime.Parse("2024-06-15").ToJson());  // "2024-06-15T00:00:00"

In [ ]:
// Null-safe — returns null for null input (no exception)
display(((object)null).ToJson());    // null
display(((string)null).ToJson());    // null

---
## 4. Chaining Examples

Value extensions compose naturally. These pipelines show realistic scenarios that combine
multiple `ValueExtensions` methods together.

### Range guard before categorisation
`IsBetween` to validate a score is in a legal range, then categorise it

In [ ]:
// Categorise a score only when it falls in the valid range [0, 100]
int[] scores = { -5, 0, 45, 75, 90, 101 };

foreach (var score in scores)
{
    if (!score.IsBetween(0, 100))
    {
        display($"{score,4}: invalid score");
        continue;
    }

    var grade = score.IsBetween(90, 100)  ? "A" :
                score.IsBetween(75, 89)   ? "B" :
                score.IsBetween(60, 74)   ? "C" : "F";

    display($"{score,4}: {grade}");
}
// -5: invalid score
//   0: F
//  45: F
//  75: B
//  90: A
// 101: invalid score

### Role check with conditional JSON serialisation
`In` to check access rights, `ToJson` to emit the audit record only when allowed

In [ ]:
var user = new { Name = "Carol", Role = "editor", LastLogin = "2024-06-15" };

bool canWrite  = user.Role.In("admin", "editor");     // True
bool canDelete = user.Role.In("admin", "superadmin"); // False

display($"Can write:  {canWrite}");   // True
display($"Can delete: {canDelete}");  // False

// Only serialise an audit record for privileged roles
var auditJson = user.Role.In("admin", "editor", "superadmin")
    ? new { user.Name, user.Role, Action = "login" }.ToJson(indentation: true)
    : null;

display(auditJson);
// {
//   "Name": "Carol",
//   "Role": "editor",
//   "Action": "login"
// }

### Serialise only values within an allowed set
`In` as a filter guard, `ToJson` for the output payload

In [ ]:
// Emit a JSON status event only for terminal states; ignore transient ones
string[] statuses = { "pending", "running", "succeeded", "failed", "cancelled" };
string[] terminalStates = { "succeeded", "failed", "cancelled" };

foreach (var status in statuses)
{
    var payload = status.In(terminalStates)
        ? new { Status = status, Timestamp = "2024-06-15T12:00:00Z" }.ToJson()
        : null;

    if (payload is not null)
        display(payload);
}
// {"Status":"succeeded","Timestamp":"2024-06-15T12:00:00Z"}
// {"Status":"failed","Timestamp":"2024-06-15T12:00:00Z"}
// {"Status":"cancelled","Timestamp":"2024-06-15T12:00:00Z"}